# DATA WRANGLING

## SETUP

In [8]:
import pandas as pd
import sqlite3
from pathlib import Path
import holidays


try:
    # if running as a .py, __file__ exists
    BASE_DIR = Path(__file__).resolve().parents[3]
except NameError:
    # if running in a notebook, __file__ does NOT exist
    BASE_DIR = Path.cwd().parents[2] 

# path to database
DATA_DIR = BASE_DIR / "TBA_project_backup" / "train_delay.db"
print(DATA_DIR)

/Users/jakoberhard/Library/CloudStorage/GoogleDrive-jakanterh@gmail.com/My Drive/uni/python/TBA_project_backup/train_delay.db


## IMPORT DATA

In [9]:
# establish SQL connection to database and load into dataframe
con = sqlite3.connect(DATA_DIR)
df_raw = pd.read_sql_query("SELECT * from train_delay", con)
con.close()

## DATA INSPECTION

In [10]:
print(df_raw.info())                      
print(df_raw.head(5))                      
print(df_raw["delay_in_min"].describe())  
# print(df_raw["train_type"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 339 entries, 0 to 338
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id                  339 non-null    object
 1   start_station_name  339 non-null    object
 2   end_station_name    339 non-null    object
 3   train_name          339 non-null    object
 4   train_type          339 non-null    object
 5   departure_planned   333 non-null    object
 6   departure_actual    333 non-null    object
 7   arrival_planned     324 non-null    object
 8   arrival_actual      324 non-null    object
 9   delay_in_min        339 non-null    int64 
dtypes: int64(1), object(9)
memory usage: 26.6+ KB
None
         id start_station_name   end_station_name train_name train_type  \
0  08000240          Mainz Hbf     Hamburg-Altona    ICE 920        ICE   
1  08000049   Braunschweig Hbf  Berlin Ostbahnhof    ICE 292        ICE   
2  08000261        München Hbf        

## DATA-TYPES

In [11]:
# preserve copy of data in df_raw
df = df_raw

# time columns
time_cols = [
    "departure_planned",
    "arrival_planned",
    "departure_actual",
    "arrival_actual"]
df[time_cols] = df[time_cols].apply(pd.to_datetime, errors='coerce')

# categorical variables
categorical_cols = [
    "start_station_name",
    "end_station_name",
    "train_name",
    "train_type"
]
df[categorical_cols] = df[categorical_cols].astype("category")

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 339 entries, 0 to 338
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id                  339 non-null    object        
 1   start_station_name  339 non-null    category      
 2   end_station_name    339 non-null    category      
 3   train_name          339 non-null    category      
 4   train_type          339 non-null    category      
 5   departure_planned   98 non-null     datetime64[ns]
 6   departure_actual    98 non-null     datetime64[ns]
 7   arrival_planned     89 non-null     datetime64[ns]
 8   arrival_actual      89 non-null     datetime64[ns]
 9   delay_in_min        339 non-null    int64         
dtypes: category(4), datetime64[ns](4), int64(1), object(1)
memory usage: 36.8+ KB


,id,start_station_name,end_station_name,train_name,train_type,departure_planned,departure_actual,arrival_planned,arrival_actual,delay_in_min
0,08000240,Mainz Hbf,Hamburg-Altona,ICE 920,ICE,2024-07-01 00:01:00,2024-07-01 00:01:00,2024-06-30 23:59:00,2024-06-30 23:59:00,0
1,08000049,Braunschweig Hbf,Berlin Ostbahnhof,ICE 292,ICE,2024-07-01 00:01:00,2024-07-01 00:01:00,2024-06-30 23:59:00,2024-06-30 23:57:00,0
2,08000261,München Hbf,Kiel Hbf,ICE 618,ICE,2024-07-01 00:01:00,2024-07-01 00:02:00,NaT,NaT,1
3,08000098,Essen Hbf,Köln Hbf,ICE 892,ICE,2024-07-01 00:02:00,2024-07-01 00:03:00,2024-07-01 00:00:00,2024-07-01 00:01:00,1
4,08002553,Hamburg-Altona,Hamburg-Altona,ICE 730,ICE,NaT,NaT,2024-07-01 00:04:00,2024-07-01 00:04:00,0


## RIDE-RELATED

### STATION NAMES, TIME AND STOPS

In [12]:
features_ride = (
    df_raw
    .groupby("train_name") # EXCHANGE BY UNIQUE IDENTIFIER
    .agg(     
        # route
        station_start=("start_station_name", "first"), # CHANGE TO CURRENT
        station_dest=("start_station_name", "last"),
        stops_total=("start_station_name", "count"),
        
        # time
        time_start_planned=("departure_planned", "first"),
        time_dest_planned=("arrival_planned", "last"),
    )
    .reset_index())

/var/folders/9n/ws1j083x5m9d1jv6n898lzlw0000gn/T/ipykernel_34190/3240452551.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("train_name") # EXCHANGE BY UNIQUE IDENTIFIER


### TRAVEL TIME

In [13]:
features_ride["travel_time"] = (
    features_ride["time_dest_planned"]
    - features_ride["time_start_planned"]
).dt.total_seconds() / 60

### CALENDAR_RELATED


In [14]:
features_ride = features_ride.assign(
    weekday=lambda d: d["time_start_planned"].dt.weekday.astype("category"),  
    weekend=lambda d: (d["weekday"].isin([5, 6])).astype("category"),
    month=lambda d: d["time_start_planned"].dt.month.astype("category")
)

### SEASON

In [15]:
# function that assigns season to month
def season(month):
    if month in [12, 1, 2]:
        return "winter"
    elif month in [3, 4, 5]:
        return "spring"
    elif month in [6, 7, 8]:
        return "summer"
    else:
        return "autumn"
    
features_ride["season"] = features_ride["month"].apply(season).astype("category")

### FEAST

In [16]:
# get holidays
de_holidays = holidays.Germany()

'''
features_ride["feast"] = (
    features_ride["time_start_planned"]
    .dt.date
    .apply(lambda d: int(d in de_holidays))
    .astype(int)
)
'''

features_ride["feast"] = (
    features_ride["time_start_planned"]
    .dt.date
    .apply(lambda d: 1 if pd.notnull(d) and d in de_holidays else 0)
)


features_ride.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 161 entries, 0 to 160
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   train_name          161 non-null    category      
 1   station_start       161 non-null    category      
 2   station_dest        161 non-null    category      
 3   stops_total         161 non-null    int64         
 4   time_start_planned  62 non-null     datetime64[ns]
 5   time_dest_planned   61 non-null     datetime64[ns]
 6   travel_time         55 non-null     float64       
 7   weekday             62 non-null     category      
 8   weekend             161 non-null    category      
 9   month               62 non-null     category      
 10  season              62 non-null     category      
 11  feast               161 non-null    int64         
dtypes: category(7), datetime64[ns](2), float64(1), int64(2)
memory usage: 35.2 KB


## STATION-RELATED

### HISTORICAL AGGREGATION

In [ ]:
# DELAY PER STATION
df = df.sort_values("ts")

g = df.groupby("station")

df["hist_avg_station"] = (
    df.groupby("station")["delay"]
      .transform(lambda s: s.shift().expanding().mean())
)

df["hist_sd_station"] = (
    df.groupby("station")["delay"]
      .transform(lambda s: s.shift().expanding().std())
)

df["hist_q90_station"] = (
    df.groupby("station")["delay"]
      .transform(lambda s: s.shift().expanding().quantile(0.9))
)


# DELAY PER STATION AND HOUR OF DAY
df["hour"] = df["ts"].dt.hour
df = df.sort_values("ts")
g = df.groupby(["station", "hour"])
df["hist_avg_delay_station_hour"] = (
    g["delay"].cumsum().shift() / g.cumcount()
)

# DELAY PER STATION AND WEEKDAY
df = df.sort_values("ts")
g = df.groupby(["station", "weekday"])
df["hist_avg_delay_station_weekday"] = (
    g["delay"].cumsum().shift() / g.cumcount()
)   

# DELAY PER STATION



### SEQUENCE-RELATED